# nb04 — Confounding gradient: Figure 6 gradient panel and Supplementary Figure S17

**Plan:** `article_1_revision_implementation_plan_261029.md`, WP6 (step 6.6) and WP9 (§9.1 Figure 6
new gradient panel, §9.2 new S17). Answers Reviewer 1, comment 1 (does batch–diagnosis confounding
drive the conclusions?).

**Experiment.** 5 RNA batches that hold all three major groups (DLBCL, FL, normal B cells; 3,404
samples). For each confounding level λ = 0, 0.1, …, 1 a 405-sample design is drawn (cap 150
samples per batch; 3 replicates), from every batch balanced (λ = 0, positive control) to one
diagnosis per batch (λ = 1, negative control). The measured bias-corrected Cramér's V (batch ×
major group) of each design is the x axis. 14 arms (12 methods + the two covariate-free variants
of SVA and ComBat; Angel and M-ComBat added after gate G4, pod run 2026-11-08) harmonize every design; the metric pod scored them. Extra designs: the uncapped
sensitivity curve (11 designs, replicate 1) and the 11 single-diagnosis batches (2,068 samples,
other batches, so it is a separate point and not on the line).

| Input | Content |
|---|---|
| `output/wp6/gradient_scores.parquet` | 630 runs (45 designs × 14 arms) with their scores (r13) |
| `output/wp6/gradient_slopes.tsv` | arm × score: mixed-model / OLS slope vs V, CI, "meaningful" flag (r13) |
| `output/wp6/gradient_failures.tsv` | runs without scores |
| `output/wp6/real_data_v.json` | real-data V: 5 gradient batches 0.398, all 7,174 samples 0.358 |
| `output/wp1/strategy_v_vs_best_scores.tsv` (+ `_spearman.tsv`) | observational inset: 14 strategies, V vs best biology scores |

**Outputs** (`figures_svg/`): `fig6_gradient_panel` (Figure 6, new panel), `figS17a_gradient_scores`,
`figS17b_gradient_slopes`, `figS17c_observational_strategies`; tables `output/wp9/nb04_*.tsv`.

**Run:** `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=collagen_3_11 nb04_confounding_gradient.ipynb` (from `article_1_revision/`).

Updated 2026-11-08 (figure review, plan `figure_review_corrections_plan_261108.md`): S17a 3 × 2 without the fold-change agreement, S17e without bands, S17c in strategy colours with code labels, bands named "mean ± SD of 3 replicate draws", every panel at its page size (180 × 275 mm page, `PANEL_SLOTS_MM`).

Updated 2026-11-07 (gate G4 review): article palettes (`method_pal`), Spearman / FDR markers in S17b, new panels S17d (λ scatter), S17e (rank trajectories), S17f (selected methods), failure table.


## §0 Configuration

### §0a Imports

In [ ]:
# Imports. revision_lib is used only for plain-word names and paths (plan WP9).
import json
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import revision_lib as rl

PN = rl.PUBLICATION_NAMES  # every label comes from here (plan §1.1)

### §0b Paths and switches

| Switch | Default | Meaning |
|---|---|---|
| `KEY_ARMS` | = `SELECTED_ARMS` | the methods of the Figure 6 gradient panel: only the methods of the 22 selected approaches (gate G4.1, Daniil 2026-11-07); no harmonization stays in S17 |
| `FIG6_SCORES` | batch removal, diagnosis retention | the two combined axes shown in Figure 6; S17 shows every score |
| `S17_SCORES` | L, N, four scaled bars | panels of S17a (3 × 2). The agreement with batch-free fold changes is no longer shown or analysed (Daniil 2026-11-08) |
| `SPREAD` | `"sd"` | band around each line: mean ± SD of the 3 replicate draws (`"sd"`, not a confidence interval) or min–max (`"range"`) |
| `SCATTER_LAMBDAS` | 0, 0.4, 1 | λ levels of the scatter panels S17d: balanced batches, the level closest to the real data (V ≈ 0.39 vs real 0.40), one diagnosis per batch |
| `SELECTED_ARMS` | SVA, ComBat, FSMVN, FSQN R, Angel, M-ComBat | gradient arms whose method is in the final selection after gate G2b. Angel and M-ComBat were added to the gradient after the visual check selected them (gate G4, 2026-11-07; pod run); they are drawn as soon as their runs are in `gradient_scores.parquet` |


In [ ]:
# Paths and switches.
WP6_DIR = rl.OUTPUT_DIR / "wp6"
WP1_DIR = rl.OUTPUT_DIR / "wp1"
FIGURES_DIR = rl.REVISION_DIR / "figures_svg"
TABLES_DIR = rl.OUTPUT_DIR / "wp9"
FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

FIG6_SCORES = ["batch_removal", "biology_retention"]
S17_SCORES = [
    "L",
    "N",
    "global_batch_scaled",
    "local_batch_scaled",
    "global_biology_scaled",
    "local_biology_scaled",
]
SPREAD = "sd"
SCATTER_LAMBDAS = [0.0, 0.4, 1.0]
SELECTED_ARMS = ["04_sva", "05_combat", "13_fsmvn", "16_fsqn_r", "25_angel", "29_combat_ref"]
REFERENCE_ARM = "01_raw"  # drawn with every subset as the reference


### §0c Palettes, plain-word names, plot settings

In [ ]:
# Article 1 palettes, copied (not imported) from the frozen figure notebook
# figures_for_article/Finally_assembled_figures_for_article.ipynb, cell 4. method_pal is
# that notebook's tab20 + tab20b palette (saturation + 0.15) over the 33 sorted methods,
# written out as hex so every revision figure uses exactly the submitted colours.
method_pal = {
    "01_raw": "#046cb4",
    "02_median_scaling": "#8bb3e8",
    "03_limma": "#ff7800",
    "04_sva": "#ffa852",
    "05_combat": "#14a014",
    "06_combat_seq": "#7cdf69",
    "07_pycombat": "#d60708",
    "08_inmoose_combatseq": "#ff7270",
    "09_ruv": "#864bbd",
    "10_mnn": "#b790d5",
    "11_harmony": "#8c4536",
    "12_scanorama": "#c48477",
    "13_fsmvn": "#e355b8",
    "14_qsmooth": "#f791bd",
    "15_fsqn_py": "#7f6c6c",
    "16_fsqn_r": "#c7a9a9",
    "17_quantile": "#bcbd06",
    "18_rank": "#dbdb6c",
    "19_tdm": "#00bccf",
    "20_shambhala": "#7cd5e5",
    "21_harmonizr": "#272979",
    "22_tmm": "#3a3ca3",
    "23_vst": "#4c50cf",
    "25_angel": "#7b7ede",
    "26_xpn": "#5d7927",
    "27_dwd": "#85a23a",
    "28_npn": "#adcf4c",
    "29_combat_ref": "#c7db7b",
    "31_ruv3prps": "#8c661c",
    "33_amdbnorm": "#bd971d",
    "34_arsyn": "#e7b02f",
    "36_explobatch": "#e7bf71",
    "38_harman": "#842925",
}
# covariate-free variants of the revision: the hue of the parent method, dashed line
method_pal["04_sva_nocov"] = method_pal["04_sva"]
method_pal["05_combat_nocov"] = method_pal["05_combat"]
strat_pal = {
    "S0_no_removal": "#a1c9f4",
    "A_confirmed_bad": "#ffb482",
    "B_extended_bad": "#8de5a1",
    "C_rnaseq_only": "#ff9f9b",
    "D_malignant_only": "#d0bbff",
    "E1_iterative_r1": "#debb9b",
    "E2_iterative_r2": "#fab0e4",
    "E3_iterative_r3": "#cfcfcf",
    "F_microarray_only": "#fffea3",
    "G_affymetrix_only": "#b9f2f0",
    "H_affymetrix_extended": "#bdb76b",
    "I_rare_batches_removed": "#b095c5",
    "J_ff_only": "#7ecbc4",
    "K_ffpe_only": "#f2c27f",
}
imp_pal = {"strict": "#2979ae", "knn": "#982d22", "softimpute": "#713689"}
post_rm_pal = {False: "#CCCCCC", True: "#333333"}

# Gradient arms in a fixed order; the colour of every arm comes from method_pal (gate G4
# review, Daniil 2026-11-07: the article palettes in every figure).
ARM_ORDER = [
    "01_raw", "03_limma", "04_sva", "04_sva_nocov", "05_combat", "05_combat_nocov",
    "10_mnn", "11_harmony", "13_fsmvn", "16_fsqn_r", "17_quantile", "33_amdbnorm",
    "25_angel", "29_combat_ref",  # added 2026-11-07 (gate G4); kept only once their runs exist
]
ARM_COLORS = {arm: method_pal[arm] for arm in ARM_ORDER}
ARM_LINESTYLES = {arm: ("--" if arm.endswith("_nocov") else "-") for arm in ARM_ORDER}


def arm_label(arm: str) -> str:
    # plain-word method name (covariate-free variants come from PUBLICATION_NAMES)
    return PN.get(arm, rl.method_display_name(arm))


In [ ]:
# Plain-word names of the scores (PUBLICATION_NAMES; scaled bars keep the bar name).
SCORE_NAMES = {
    "L": PN["L"],
    "N": PN["N"],
    "batch_removal": PN["batch_removal"],
    "biology_retention": PN["biology_retention"],
}
for bar in rl.FOUR_SCORE_BARS:
    SCORE_NAMES[f"{bar}_scaled"] = PN[bar] + " (scaled)"
V_LABEL = PN["cramers_v"]
REAL_V_COLOR = "#B22222"  # warm: real (tumour-containing) data position

In [ ]:
# Plot settings (SVG + PDF + PNG 200 dpi) and helpers. Font sizes, slot_figsize and
# save_figure are redefined in Figma units by the next cell (Daniil 2026-11-09).
GLOBAL_FONT_SIZE = 7
SMALL_FONT_SIZE = 6  # smallest text on any panel (journal minimum)
plt.rcParams.update({
    "font.size": GLOBAL_FONT_SIZE,
    "axes.labelsize": GLOBAL_FONT_SIZE,
    "axes.titlesize": GLOBAL_FONT_SIZE + 1,
    "xtick.labelsize": GLOBAL_FONT_SIZE - 1,
    "ytick.labelsize": GLOBAL_FONT_SIZE - 1,
    "legend.fontsize": GLOBAL_FONT_SIZE - 1,
    "pdf.fonttype": "truetype",
    "svg.fonttype": "none",
    "figure.dpi": 100,
})
sns.set_style("ticks")


# Panel slots on the full figure page (D2, Daniil 2026-11-08). The page is the Figma frame
# "Figure 4" (node 761-28, 753 × 1150 px) = the 180 × 275 mm text area; every panel is drawn
# at its true size on that page, so GLOBAL_FONT_SIZE is the printed size in pt (≥ 6 pt).
# Copied from figures_svg/PLACEMENT.md (column "slot (mm)"); same dict in nb03–nb05.
PAGE_MM = (180, 275)
PANEL_SLOTS_MM = {
    "fig6B_eta_squared": (85, 50),
    "fig6_gradient_panel": (95, 50),
    "fig6CD_four_scores": (180, 110),
    "fig6F_decision_tree": (180, 110),
    "figS14a_marker_correlation_gate": (180, 85),
    "figS14b_prediction_gate": (180, 85),
    "figS14c_label_leakage": (180, 80),
    "figS16a_robustness_tau": (180, 140),
    "figS16b_front_stability": (180, 115),
    "figS17a_gradient_scores": (180, 125),
    "figS17b_gradient_slopes": (180, 55),
    "figS17c_observational_strategies": (180, 45),
    "figS17d_gradient_scatter": (180, 58),
    "figS17e_gradient_rank_trajectories": (180, 62),
    "figS17f_selected_methods_scores": (180, 105),
    "figS18a_v1_rank_on_test": (180, 50),  # half height (Daniil 2026-11-09)
    "figS18b_v1_regret": (180, 90),  # was S18c (S18b merged into S18a, 2026-11-09)
    "figS2_merged_distributions_all_cohorts": (180, 275),
}
MM_PER_INCH = 25.4


def slot_figsize(stem: str) -> tuple[float, float]:
    """Figure size in inches of the page slot of a panel file (PANEL_SLOTS_MM)."""
    width_mm, height_mm = PANEL_SLOTS_MM[stem]
    return width_mm / MM_PER_INCH, height_mm / MM_PER_INCH


def save_figure(fig: plt.Figure, stem: str) -> None:
    # one file per panel: SVG + PDF + PNG (200 dpi)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    print(f"saved figures_svg/{stem}.svg/.pdf/.png")


def save_table(table: pd.DataFrame, stem: str) -> None:
    # every plotted table is kept next to the figure inputs
    table.to_csv(TABLES_DIR / f"{stem}.tsv", sep="\t", index=False)
    print(f"saved output/wp9/{stem}.tsv ({len(table)} rows)")

In [ ]:
# Figma units (Daniil 2026-11-09, `daniil_review_261109.md`). Figma imports an SVG with
# 1 SVG pt = 4/3 px, and the figure frame (Figma node 1607-116664) is 756 × 1159 px for the
# 180 × 275 mm page. So every panel is drawn at (slot in Figma px) × 0.75 pt, and every text
# is 7.5 pt = 10 px in Figma. Legends and colour bars ("palettes") may stick out of the slot:
# only the plot area (axes with their tick labels and titles) is fitted to it.
import matplotlib.text as mtext
from matplotlib.transforms import Bbox

FIGMA_FRAME_PX = (756, 1159)
FIGMA_PAGE_MM = (180, 275)
FIGMA_PX_PER_PT = 4 / 3
FIGMA_PX_PER_MM = FIGMA_FRAME_PX[0] / FIGMA_PAGE_MM[0]  # 4.2 px per mm of the page
FIGMA_FONT_PX = 10
FIGMA_FONT_PT = FIGMA_FONT_PX / FIGMA_PX_PER_PT  # 7.5 pt
FIGMA_FIT_LOG = rl.OUTPUT_DIR / "wp14" / "figma_fit" / "nb04_confounding_gradient.tsv"
FIGMA_FIT_LOG.parent.mkdir(parents=True, exist_ok=True)

GLOBAL_FONT_SIZE = FIGMA_FONT_PT
SMALL_FONT_SIZE = FIGMA_FONT_PT
plt.rcParams.update({key: FIGMA_FONT_PT for key in (
    "font.size", "axes.labelsize", "axes.titlesize", "xtick.labelsize", "ytick.labelsize",
    "legend.fontsize", "legend.title_fontsize", "figure.titlesize")})


def mm_figsize(width_mm: float, height_mm: float) -> tuple[float, float]:
    """Figure size in inches that lands in Figma at the slot size (page mm → Figma px)."""
    to_inches = FIGMA_PX_PER_MM / FIGMA_PX_PER_PT / 72
    return width_mm * to_inches, height_mm * to_inches


def slot_figsize(stem: str) -> tuple[float, float]:
    """Figure size in inches of the page slot of a panel file (PANEL_SLOTS_MM)."""
    return mm_figsize(*PANEL_SLOTS_MM[stem])


def batch_display_name(batch: str) -> str:
    """Plain batch name: underscores → spaces; microarrays drop the last "Unknown" (library
    preparation is never known for a microarray), RNA-seq keeps its library type."""
    parts = str(batch).split("_")
    if parts[0] != "RNASeq" and len(parts) > 2 and parts[-1] == "Unknown":
        parts = parts[:-1]
    return " ".join(parts)


def is_palette_axis(ax: plt.Axes) -> bool:
    """Colour bars and legend-only axes are palettes: Daniil moves them by hand in Figma."""
    if hasattr(ax, "_colorbar") or getattr(ax, "_figma_palette", False):
        return True
    has_data = bool(ax.lines or ax.collections or ax.images or ax.patches or ax.texts)
    return not ax.axison and not has_data


def plot_extent_px(fig: plt.Figure) -> tuple[float, float]:
    """Width and height in Figma px of the plot area (axes, tick labels, titles, texts on
    the axes, suptitle), without legends and colour bars."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    boxes = [ax.get_tightbbox(renderer, bbox_extra_artists=list(ax.texts))
             for ax in fig.axes if ax.get_visible() and not is_palette_axis(ax)]
    boxes += [t.get_window_extent(renderer) for t in fig.texts if t.get_text()]
    if getattr(fig, "_suptitle", None) is not None and fig._suptitle.get_text():
        boxes.append(fig._suptitle.get_window_extent(renderer))
    boxes = [b for b in boxes if b is not None and b.width > 0]
    # a legend-only figure has no plot area: measure the whole figure
    union = Bbox.union(boxes) if boxes else fig.bbox
    px_per_display = 72 / fig.dpi * FIGMA_PX_PER_PT
    return union.width * px_per_display, union.height * px_per_display


def set_all_fonts(fig: plt.Figure, size: float) -> None:
    """Every text of the figure at one size (10 px in Figma)."""
    for ax in fig.axes:
        ax.tick_params(labelsize=size)
    for text in fig.findobj(mtext.Text):
        if text.get_text():
            text.set_fontsize(size)


def fit_to_slot(fig: plt.Figure, width_mm: float, height_mm: float | None,
                n_iterations: int = 8) -> None:
    """Resize the figure until its plot area is width_mm wide (and height_mm high; None keeps
    the current aspect ratio of the plot area). Fonts keep their size, so a few redraws are
    needed; fixed-aspect plots (squares) only shrink to fit."""
    target_w = width_mm * FIGMA_PX_PER_MM
    for _ in range(n_iterations):
        plot_w, plot_h = plot_extent_px(fig)
        target_h = plot_h * target_w / plot_w if height_mm is None else height_mm * FIGMA_PX_PER_MM
        if abs(plot_w - target_w) < 1 and abs(plot_h - target_h) < 1:
            break
        fig_w, fig_h = fig.get_size_inches()
        fig.set_size_inches(max(fig_w * target_w / plot_w, 0.5), max(fig_h * target_h / plot_h, 0.5))
    plot_w, plot_h = plot_extent_px(fig)
    target_h = FIGMA_FRAME_PX[1] if height_mm is None else height_mm * FIGMA_PX_PER_MM
    shrink = min(1.0, target_w / plot_w, target_h / plot_h)
    if shrink < 0.995:  # fixed-aspect plots grew in the other direction: scale the whole panel
        fig.set_size_inches(*(fig.get_size_inches() * shrink))


def log_fit(stem: str, fig: plt.Figure, slot_mm: tuple | None) -> None:
    """One row per saved panel: slot and plot area in Figma px (read by r97_figma_check.py)."""
    plot_w, plot_h = plot_extent_px(fig)
    row = pd.DataFrame([{
        "stem": stem,
        "slot_w_px": None if slot_mm is None else round(slot_mm[0] * FIGMA_PX_PER_MM, 1),
        "slot_h_px": None if slot_mm is None or slot_mm[1] is None else round(slot_mm[1] * FIGMA_PX_PER_MM, 1),
        "plot_w_px": round(plot_w, 1), "plot_h_px": round(plot_h, 1),
    }])
    old = pd.read_csv(FIGMA_FIT_LOG, sep="\t") if FIGMA_FIT_LOG.exists() else row.iloc[:0]
    pd.concat([old[old["stem"] != stem], row]).to_csv(FIGMA_FIT_LOG, sep="\t", index=False)


def save_figure(fig: plt.Figure, stem: str, slot_mm: tuple | None = None,
                keep_fonts: bool = False) -> None:
    """Fonts → 10 Figma px (unless keep_fonts: large clustermaps), plot area → slot (default:
    PANEL_SLOTS_MM[stem] when the notebook has it), then SVG + PDF + PNG (200 dpi)."""
    if slot_mm is None and stem in globals().get("PANEL_SLOTS_MM", {}):
        slot_mm = PANEL_SLOTS_MM[stem]
    if not keep_fonts:
        set_all_fonts(fig, FIGMA_FONT_PT)
    if slot_mm is not None:
        fit_to_slot(fig, *slot_mm)
    log_fit(stem, fig, slot_mm)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    print(f"saved figures_svg/{stem}.svg/.pdf/.png")


## §1 Load the gradient runs

Expected: 630 runs (462 capped = 11 λ × 3 replicates × 14 arms; 154 uncapped = 11 × 14; 14 on the
single-diagnosis control), 619 with scores, 11 failures (ComBat and M-ComBat with the diagnosis
covariate cannot fit a design where diagnosis and batch coincide; AMDBNorm 6 h timeout on the
2,068-sample control). Angel and M-ComBat were added after gate G4 (pod run 2026-11-08).

In [ ]:
# Load runs, failures and the real-data V.
gradient_scores = pd.read_parquet(WP6_DIR / "gradient_scores.parquet")
gradient_failures = pd.read_csv(WP6_DIR / "gradient_failures.tsv", sep="\t")
real_data_v = json.loads((WP6_DIR / "real_data_v.json").read_text())
gradient_scores["method_label"] = gradient_scores["arm"].map(arm_label)
completed_runs = gradient_scores[gradient_scores["status"] == "ok"]

print(gradient_scores.groupby(["kind", "cap"]).size().to_string())
print(f"runs with scores: {len(completed_runs)} of {len(gradient_scores)}")
print(gradient_failures[["arm", "kind", "lambda", "replicate", "cap", "status"]].to_string(index=False))
print("real-data V:", {k: round(v, 3) for k, v in real_data_v.items() if isinstance(v, float)})
assert len(gradient_scores) == 45 * gradient_scores["arm"].nunique()
# Angel / M-ComBat appear once their pod run is collected; every arm in the data must be known.
assert set(gradient_scores["arm"]) <= set(ARM_ORDER), set(gradient_scores["arm"]) - set(ARM_ORDER)
ARM_ORDER = [arm for arm in ARM_ORDER if arm in set(gradient_scores["arm"])]
SELECTED_ARMS = [arm for arm in SELECTED_ARMS if arm in ARM_ORDER]
KEY_ARMS = SELECTED_ARMS  # Figure 6 gradient panel: selected methods only (G4.1)
print("arms in the data:", len(ARM_ORDER), "; selected arms drawn:", [arm_label(a) for a in SELECTED_ARMS])

### §1b Mean and spread over the replicates

For every arm × λ level of the capped gradient: mean V, mean score and its spread over the 3
replicate draws. The band drawn around each line is **mean ± SD of the 3 replicate draws** (not a
confidence interval; Daniil's question 2026-11-08). The only confidence intervals of this notebook
are the OLS / mixed-model 95 % CIs of the slopes (S17b). The λ = 0 and λ = 1 rows are the positive and negative controls.

In [ ]:
# Summarise the capped gradient: arm × λ → mean V, mean / SD / min / max of each score.
capped_runs = completed_runs[(completed_runs["kind"] == "gradient") & (completed_runs["cap"] == "150")]
all_scores = sorted(set(S17_SCORES) | set(FIG6_SCORES))
gradient_summary = (
    capped_runs.melt(
        id_vars=["arm", "lambda", "replicate", "cramers_v_major_group"],
        value_vars=all_scores, var_name="score", value_name="value",
    )
    .groupby(["arm", "score", "lambda"])
    .agg(
        v_mean=("cramers_v_major_group", "mean"),
        value_mean=("value", "mean"),
        value_sd=("value", "std"),
        value_min=("value", "min"),
        value_max=("value", "max"),
        n_replicates=("value", "count"),
    )
    .reset_index()
)
uncapped_runs = completed_runs[(completed_runs["kind"] == "gradient") & (completed_runs["cap"] == "none")]
single_dx_runs = completed_runs[completed_runs["kind"] == "single_dx_control"]
save_table(gradient_summary, "nb04_gradient_summary")
print(gradient_summary.groupby("arm")["n_replicates"].agg(["min", "max"]).to_string())

### §0d Plot helper

In [ ]:
# Draw one score vs V for a list of arms: mean ± spread over the replicates.
import textwrap  # long score names wrapped on the axis


def draw_gradient(ax, score: str, arms: list[str], show_uncapped: bool, show_single_dx: bool) -> None:
    for arm in arms:
        rows = gradient_summary[(gradient_summary["arm"] == arm) & (gradient_summary["score"] == score)]
        rows = rows.sort_values("v_mean")
        if rows.empty:
            continue
        if SPREAD == "sd":
            low, high = rows["value_mean"] - rows["value_sd"].fillna(0), rows["value_mean"] + rows["value_sd"].fillna(0)
        else:
            low, high = rows["value_min"], rows["value_max"]
        ax.plot(rows["v_mean"], rows["value_mean"], color=ARM_COLORS[arm], linestyle=ARM_LINESTYLES[arm],
                marker="o", markersize=3, linewidth=1.4, label=arm_label(arm))
        ax.fill_between(rows["v_mean"], low, high, color=ARM_COLORS[arm], alpha=0.12, linewidth=0)
        if show_uncapped:
            extra = uncapped_runs[uncapped_runs["arm"] == arm].sort_values("cramers_v_major_group")
            ax.plot(extra["cramers_v_major_group"], extra[score], color=ARM_COLORS[arm],
                    linestyle=":", linewidth=0.9, alpha=0.8)
        if show_single_dx:
            point = single_dx_runs[single_dx_runs["arm"] == arm]
            ax.scatter(point["cramers_v_major_group"], point[score], marker="X", s=40,
                       color=ARM_COLORS[arm], edgecolor="white", linewidth=0.4, zorder=4)
    ax.axvline(real_data_v["five_batches"], color=REAL_V_COLOR, linestyle="--", linewidth=0.9)
    ax.axvline(real_data_v["all_samples"], color=REAL_V_COLOR, linestyle=":", linewidth=0.9)
    ax.set_xlim(-0.03, 1.05)
    ax.set_xlabel(V_LABEL)
    ax.set_ylabel(textwrap.fill(SCORE_NAMES[score], 30))
    sns.despine(ax=ax)


def band_handle() -> mpatches.Patch:
    # legend entry of the grey band: what the band is (Daniil 2026-11-08: not a CI)
    return mpatches.Patch(color="#999999", alpha=0.3, label="Band: mean ± SD of 3 replicate draws")


def real_v_handles() -> list:
    # legend entries of the two real-data positions
    return [
        mlines.Line2D([], [], color=REAL_V_COLOR, linestyle="--",
                      label=f"Real data, 5 gradient batches (V = {real_data_v['five_batches']:.2f})"),
        mlines.Line2D([], [], color=REAL_V_COLOR, linestyle=":",
                      label=f"Real data, all 7,174 samples (V = {real_data_v['all_samples']:.2f})"),
    ]

## §2 Figure 6, new panel: the gradient for the key methods

Two combined axes (batch removal, diagnosis retention; mean of the scaled global and local bars)
against V for the methods of the decision tree, mean ± SD over 3 replicates. Dashed red line = the
real data on the same 5 batches; dotted = all 7,174 samples.

In [ ]:
# Figure 6 gradient panel: one subplot per combined axis.
fig, axes = plt.subplots(1, len(FIG6_SCORES), figsize=slot_figsize("fig6_gradient_panel"), sharex=True)
for ax, score in zip(axes, FIG6_SCORES):
    draw_gradient(ax, score, KEY_ARMS, show_uncapped=False, show_single_dx=False)
    ax.set_xlabel("Cramér's V\n(batch × diagnosis)")  # narrow main-figure panel
handles, labels = axes[0].get_legend_handles_labels()
extra = real_v_handles() + [band_handle()]
fig.legend(handles + extra, labels + [h.get_label() for h in extra],
           loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=2, frameon=False, fontsize=SMALL_FONT_SIZE)
fig.tight_layout()
save_figure(fig, "fig6_gradient_panel")
plt.show()

## §3 Supplementary Figure S17a: every arm × every score

Solid / dashed lines: capped gradient (dashed = covariate-free variant); dotted: uncapped
sensitivity curve (replicate 1); ×: the 11-single-diagnosis-batch control (different batches, not
on the line). L of no harmonization, limma and FSMVN is 1 by construction: L is the within-cohort
Spearman ρ of each marker gene against the unharmonized data, and a per-batch location / scale
shift keeps the ranks inside a cohort.

In [ ]:
# S17a: 3 × 2 grid of scores, all 14 arms.
fig, axes = plt.subplots(3, 2, figsize=slot_figsize("figS17a_gradient_scores"), sharex=True)
for ax, score in zip(axes.flat, S17_SCORES):
    draw_gradient(ax, score, ARM_ORDER, show_uncapped=True, show_single_dx=True)
    ax.tick_params(labelbottom=True)
handles, labels = axes.flat[0].get_legend_handles_labels()
extra_handles = real_v_handles() + [band_handle(), 
    mlines.Line2D([], [], color="#555555", linestyle=":", label="Uncapped designs (replicate 1)"),
    mlines.Line2D([], [], color="#555555", marker="X", linestyle="none", label=PN["L11_single_dx"]),
]
fig.legend(handles + extra_handles, labels + [h.get_label() for h in extra_handles],
           loc="upper center", bbox_to_anchor=(0.5, 0.005), ncol=3, frameon=False, fontsize=SMALL_FONT_SIZE)
fig.tight_layout()
save_figure(fig, "figS17a_gradient_scores")
plt.show()

## §3b Supplementary Figure S17d: batch removal vs diagnosis retention at three λ levels

Approved at gate G4 (Daniil, 2026-11-07). One dot per method (median over the 3 replicates of the
capped designs) at λ = 0 (balanced), λ = 0.4 (V ≈ 0.39, the level of the real data on these 5
batches, V = 0.40) and λ = 1 (one diagnosis per batch). Large labelled dots = the Pareto front of
the panel (no other method is better on both axes); small dots = the rest. It shows how the best
methods move as confounding grows.

In [ ]:
# Median of the two combined axes per arm and λ, Pareto front inside each λ panel.
scatter_points = (
    capped_runs[capped_runs["lambda"].isin(SCATTER_LAMBDAS)]
    .groupby(["lambda", "arm"])[["batch_removal", "biology_retention", "cramers_v_major_group"]]
    .median()
    .reset_index()
)


def pareto_mask(points: pd.DataFrame) -> pd.Series:
    # True when no other point is at least as good on both axes and better on one
    values = points[["batch_removal", "biology_retention"]].to_numpy()
    dominated = [
        any((other >= value).all() and (other > value).any() for other in values)
        for value in values
    ]
    return pd.Series(~np.array(dominated), index=points.index)


scatter_points["on_front"] = pd.concat([pareto_mask(part) for _, part in scatter_points.groupby("lambda")])
scatter_points["method_label"] = scatter_points["arm"].map(arm_label)
save_table(scatter_points, "nb04_gradient_scatter_points")
print(scatter_points[scatter_points["on_front"]][["lambda", "method_label"]].groupby("lambda")["method_label"].apply(list).to_string())

In [ ]:
# S17d: three scatter panels sharing both axes; front labels placed by adjustText (no overlap).
from adjustText import adjust_text

SHORT_LABELS = {"04_sva_nocov": "SVA (no covariate)", "05_combat_nocov": "ComBat (no covariate)"}
LABEL_MERGE_DISTANCE = 0.03  # front points this close (|Δx| + |Δy|) share one label
fig, axes = plt.subplots(1, len(SCATTER_LAMBDAS), figsize=slot_figsize("figS17d_gradient_scatter"), sharex=True, sharey=True)
for ax, lam in zip(axes, SCATTER_LAMBDAS):
    panel = scatter_points[scatter_points["lambda"] == lam]
    front_points = panel[panel["on_front"]].sort_values("biology_retention", ascending=False)
    label_groups = []  # front points closer than LABEL_MERGE_DISTANCE share one label
    for row in front_points.itertuples():
        for group in label_groups:
            if abs(group["x"] - row.batch_removal) + abs(group["y"] - row.biology_retention) < LABEL_MERGE_DISTANCE:
                group["names"].append(SHORT_LABELS.get(row.arm, row.method_label))
                break
        else:
            label_groups.append({"x": row.batch_removal, "y": row.biology_retention, "names": [SHORT_LABELS.get(row.arm, row.method_label)]})
    texts = [ax.text(group["x"], group["y"], textwrap.fill(", ".join(group["names"]), 22), fontsize=SMALL_FONT_SIZE)
             for group in label_groups]
    for row in panel.itertuples():
        ax.scatter(row.batch_removal, row.biology_retention, s=45 if row.on_front else 14,
                   color=ARM_COLORS[row.arm], edgecolor="black" if row.on_front else "none",
                   linewidth=0.8, marker="s" if row.arm.endswith("_nocov") else "o", zorder=3)
    adjust_text(texts, ax=ax, expand=(1.6, 1.8), force_text=(0.6, 0.8), iter_lim=2000,
                arrowprops={"arrowstyle": "-", "color": "#999999", "linewidth": 0.5})
    v_level = panel["cramers_v_major_group"].median()
    ax.set_title(f"λ = {lam:.1f} (V = {v_level:.2f})")
    ax.set_xlabel(PN["batch_removal"])
    ax.grid(color="#EEEEEE", linewidth=0.6)
    sns.despine(ax=ax)
axes[0].set_ylabel(PN["biology_retention"])
handles = [mlines.Line2D([], [], marker="s" if arm.endswith("_nocov") else "o", linestyle="none",
                         color=ARM_COLORS[arm], label=arm_label(arm)) for arm in ARM_ORDER]
handles.append(mlines.Line2D([], [], marker="o", linestyle="none", markersize=6, markerfacecolor="white",
                             markeredgecolor="black", label="Pareto front (labelled)"))
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=5, frameon=False, fontsize=SMALL_FONT_SIZE)
fig.tight_layout()
save_figure(fig, "figS17d_gradient_scatter")
plt.show()

## §3c Supplementary Figure S17e: rank of each method along the gradient

Gate G4 review (Daniil, 2026-11-07): "how do the best approaches move as confounding grows?".
Inside every gradient design `r13_gradient_stats.py` gates the 14 arms (marker correlation ρ ≥ 0.7;
prediction F1 not worse than no harmonization − 0.05 and above its permutation null) and ranks them
in the order of the selection rule (candidates first, Pareto layer on batch removal × diagnosis
retention, then local batch mixing), 1 = best (`output/wp6/gradient_ranks.tsv`). Line = median rank
over the 3 replicates; no band on either half (Daniil 2026-11-08: the left-half bands were messy), so
the two halves differ only in the set of methods. Left: the methods of the final selection that are in the
gradient (SVA, ComBat, FSMVN, FSQN R, Angel, M-ComBat) with no harmonization as reference; right: all 14
arms. Angel and M-ComBat were added after gate G4 (pod run 2026-11-08).

In [ ]:
# Median / min / max rule rank per arm and λ (capped designs).
gradient_ranks = pd.read_csv(WP6_DIR / "gradient_ranks.tsv", sep="\t")
capped_ranks = gradient_ranks[gradient_ranks["cap"].astype(str) == "150"]
rank_summary = (
    capped_ranks.groupby(["arm", "lambda"])
    .agg(v_mean=("cramers_v_major_group", "mean"), rank_median=("rule_rank", "median"),
         rank_min=("rule_rank", "min"), rank_max=("rule_rank", "max"), n_ranked=("n_ranked", "max"),
         candidate_share=("candidate", "mean"))
    .reset_index()
)
save_table(rank_summary, "nb04_gradient_rank_trajectories")
print(rank_summary.pivot_table(index="arm", columns="lambda", values="rank_median").rename(index=arm_label).round(1).to_string())
assert rank_summary["n_ranked"].max() == len(ARM_ORDER)

In [ ]:
# S17e: rank trajectories, selected methods (left) and all arms (right).
def draw_rank_trajectories(ax, arms: list[str]) -> None:
    for arm in arms:
        rows = rank_summary[rank_summary["arm"] == arm].sort_values("v_mean")
        ax.plot(rows["v_mean"], rows["rank_median"], color=ARM_COLORS[arm], linestyle=ARM_LINESTYLES[arm],
                marker="o", markersize=2.5, linewidth=1.5 if arm in SELECTED_ARMS + [REFERENCE_ARM] else 1.1,
                label=arm_label(arm))
    ax.axvline(real_data_v["five_batches"], color=REAL_V_COLOR, linestyle="--", linewidth=0.9)
    ax.set_ylim(len(ARM_ORDER) + 0.5, 0.5)  # rank 1 on top
    ax.set_yticks(range(1, len(ARM_ORDER) + 1, 2))
    ax.set_xlabel(V_LABEL)
    ax.set_ylabel("Rank under the selection rule (1 = best)")
    ax.grid(axis="y", color="#EEEEEE", linewidth=0.6)
    sns.despine(ax=ax)


fig, axes = plt.subplots(1, 2, figsize=slot_figsize("figS17e_gradient_rank_trajectories"), sharey=True)
draw_rank_trajectories(axes[0], [REFERENCE_ARM] + SELECTED_ARMS)
axes[0].set_title("Methods of the final selection")
draw_rank_trajectories(axes[1], ARM_ORDER)
axes[1].set_title("All methods of the gradient")
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles + real_v_handles()[:1], labels + [real_v_handles()[0].get_label()],
           loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=3, frameon=False, fontsize=SMALL_FONT_SIZE)
fig.tight_layout()
save_figure(fig, "figS17e_gradient_rank_trajectories")
plt.show()

## §3d Supplementary Figure S17f: the scores of the selected methods along the gradient

The same data as S17a, restricted to no harmonization and the four selected methods, so the
trajectories can be read one by one: the four scaled score bars, marker correlation (L) and
cross-batch prediction (N), mean ± SD over the 3 replicates.

In [ ]:
# S17f: 2 × 3 grid, selected methods only.
selected_scores = ["global_batch_scaled", "local_batch_scaled", "L", "global_biology_scaled", "local_biology_scaled", "N"]
fig, axes = plt.subplots(2, 3, figsize=slot_figsize("figS17f_selected_methods_scores"), sharex=True)
for ax, score in zip(axes.flat, selected_scores):
    draw_gradient(ax, score, [REFERENCE_ARM] + SELECTED_ARMS, show_uncapped=False, show_single_dx=False)
    ax.tick_params(labelbottom=True)
    ax.grid(color="#F0F0F0", linewidth=0.6)
handles, labels = axes.flat[0].get_legend_handles_labels()
extra = real_v_handles() + [band_handle()]
fig.legend(handles + extra, labels + [h.get_label() for h in extra],
           loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4, frameon=False, fontsize=SMALL_FONT_SIZE)
fig.tight_layout()
save_figure(fig, "figS17f_selected_methods_scores")
plt.show()

## §3e Failed runs (reported in the Methods)

Gate G4 review (Daniil, 2026-11-07): report every failure with a simple description of the error.

In [ ]:
# Failed gradient runs with a plain-word reason.
FAILURE_REASONS = {  # run status of the harmonization → plain description (Methods paragraph, stage B)
    "failed": "the method stopped with an error: with one diagnosis per batch the diagnosis covariate is identical to the batch, so the design matrix is singular",
    "timeout": "the run exceeded the 6-hour limit",
    "method_returned_input": "the method returned its input unchanged",
}
run_status = gradient_scores[["subset_id", "arm", "run_status"]]  # failed / timeout of the harmonization itself
failure_report = gradient_failures.merge(run_status, on=["subset_id", "arm"], how="left")
failure_report = failure_report.assign(
    method_label=failure_report["arm"].map(arm_label),
    reason=failure_report["run_status"].map(FAILURE_REASONS),
)
save_table(failure_report, "nb04_gradient_failures")
print(failure_report[["method_label", "kind", "lambda", "replicate", "cap", "run_status", "reason"]].to_string(index=False))
assert failure_report["reason"].notna().all(), "a failure status without a plain-word reason"

## §4 Supplementary Figure S17b: slopes against V

Slope of score ~ V + (1 | replicate) (ordinary least squares when the mixed model cannot be fitted),
with 95 % CI. Filled marker = the score changes with confounding by **both** criteria: Spearman ρ
(score vs V over the 33 runs) with Benjamini–Hochberg FDR < 0.05 over every arm × score test
(gate G4 review, Daniil 2026-11-07), **and** the effect-size criterion fixed before the results
(|mean(λ = 1) − mean(λ = 0)| > 2 × replicate SD at λ = 0). Hollow = not both, or not evaluable
(λ = 1 failed, or the score is constant). The constant L arms have no slope.


In [ ]:
# Load the slopes and draw a forest plot per score.
gradient_slopes = pd.read_csv(WP6_DIR / "gradient_slopes.tsv", sep="\t")
forest_scores = ["L", "N", "batch_removal", "biology_retention"]
forest_table = gradient_slopes[gradient_slopes["score"].isin(forest_scores)].copy()
forest_table["method_label"] = forest_table["arm"].map(arm_label)
save_table(forest_table, "nb04_gradient_slopes_forest")

fig, axes = plt.subplots(1, len(forest_scores), figsize=slot_figsize("figS17b_gradient_slopes"), sharey=True)
y_positions = {arm: i for i, arm in enumerate(reversed(ARM_ORDER))}
for ax, score in zip(axes, forest_scores):
    rows = forest_table[forest_table["score"] == score]
    for row in rows.itertuples():
        if pd.isna(row.slope):
            continue
        y = y_positions[row.arm]
        ax.plot([row.slope_ci_low, row.slope_ci_high], [y, y], color=ARM_COLORS[row.arm], linewidth=1.2)
        ax.scatter(row.slope, y, s=36, color=ARM_COLORS[row.arm] if row.changes_with_confounding else "white",
                   edgecolor=ARM_COLORS[row.arm], linewidth=1.2, zorder=3)
    ax.axvline(0, color="#999999", linewidth=0.8)
    ax.set_title(textwrap.fill(SCORE_NAMES[score], 24), fontsize=SMALL_FONT_SIZE)
    ax.set_xlabel("Slope per unit of V\n(95 % CI)")
    ax.xaxis.set_major_locator(plt.MaxNLocator(3))
    sns.despine(ax=ax)
axes[0].set_yticks(list(y_positions.values()))
axes[0].set_yticklabels([arm_label(arm) for arm in y_positions], fontsize=SMALL_FONT_SIZE)
fig.legend(handles=[
    mlines.Line2D([], [], marker="o", linestyle="none", color="#555555", label="Spearman FDR < 0.05 and change > 2 × replicate SD"),
    mlines.Line2D([], [], marker="o", linestyle="none", markerfacecolor="white", markeredgecolor="#555555", label="Not both"),
], loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=2, frameon=False)
fig.tight_layout()
save_figure(fig, "figS17b_gradient_slopes")
plt.show()
print(forest_table.pivot_table(index="method_label", columns="score", values="spearman_rho").round(2).to_string())
print(forest_table.pivot_table(index="method_label", columns="score", values="changes_with_confounding").to_string())

## §5 Supplementary Figure S17c: observational inset (14 strategies)

WP1 step 1.4: across the 14 sample-removal strategies of the main benchmark, does the measured
confounding (bias-corrected V, batch × diagnosis) predict the best biology score reached on that
strategy? Spearman ρ, n = 14 (low power). Dots in the strategy colours of Figure 2A (`strat_pal`),
labelled with the strategy code (S0, A … K; the legend gives code · name); labels are placed by
`adjustText` and the cell asserts that no two labels overlap.

In [ ]:
# Observational inset: strategy V vs the best scaled diagnosis bars.
# Dots in the strategy colours of Figure 2A (strat_pal), labelled with the strategy code only;
# the legend maps code → name. Labels are placed by adjustText and checked for overlap.
from adjustText import adjust_text

strategy_points = pd.read_csv(WP1_DIR / "strategy_v_vs_best_scores.tsv", sep="\t").rename(columns={"Unnamed: 0": "strat"})
strategy_spearman = pd.read_csv(WP1_DIR / "strategy_v_vs_best_scores_spearman.tsv", sep="\t").set_index("score")
assert set(strategy_points["strat"]) <= set(strat_pal), set(strategy_points["strat"]) - set(strat_pal)
inset_scores = ["global_biology_scaled", "local_biology_scaled"]


def label_boxes_overlap(fig, texts) -> list[tuple[str, str]]:
    """Pairs of label texts whose rendered bounding boxes overlap."""
    renderer = fig.canvas.get_renderer()
    boxes = [(text.get_text(), text.get_window_extent(renderer)) for text in texts]
    return [(a, b) for i, (a, box_a) in enumerate(boxes) for b, box_b in boxes[i + 1:] if box_a.overlaps(box_b)]


fig, axes = plt.subplots(1, 2, figsize=slot_figsize("figS17c_observational_strategies"), sharex=True)
for ax, score in zip(axes, inset_scores):
    ax.scatter(strategy_points["V_batch_x_diagnosis"], strategy_points[score], s=50,
               color=strategy_points["strat"].map(strat_pal), edgecolor="#333333", linewidth=0.6, zorder=3)
    stats = strategy_spearman.loc[score]
    ax.set_title(f"Spearman ρ = {stats['spearman_rho']:.2f}, p = {stats['pvalue']:.2f}, n = {int(stats['n_strategies'])}",
                 fontsize=GLOBAL_FONT_SIZE)
    ax.set_xlabel(V_LABEL)
    ax.set_ylabel(textwrap.fill("Best " + SCORE_NAMES[score][0].lower() + SCORE_NAMES[score][1:], 24))
    sns.despine(ax=ax)
handles = [mlines.Line2D([], [], marker="o", linestyle="none", markerfacecolor=strat_pal[s], markeredgecolor="#333333",
                         label=rl.STRATEGY_DISPLAY_NAMES[s]) for s in strategy_points["strat"]]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4, frameon=False, fontsize=SMALL_FONT_SIZE)
fig.tight_layout()
# labels are placed after the panel has its final (slot) size, so the fit cannot move them
set_all_fonts(fig, FIGMA_FONT_PT)
fit_to_slot(fig, *PANEL_SLOTS_MM["figS17c_observational_strategies"])
for ax, score in zip(axes, inset_scores):
    texts = [ax.text(row.V_batch_x_diagnosis, getattr(row, score), rl.STRATEGY_CODES[row.strat], fontsize=SMALL_FONT_SIZE)
             for row in strategy_points.itertuples()]
    # 10 px labels (2026-11-09) need more room: widen the repulsion until no two labels touch
    for expand in [(1.6, 1.9), (2.2, 2.6), (3.0, 3.4), (4.0, 4.4)]:
        adjust_text(texts, ax=ax, expand=expand, force_text=(0.8, 1.0), iter_lim=2000,
                    objects=ax.collections[0],
                    arrowprops={"arrowstyle": "-", "color": "#999999", "linewidth": 0.5})
        fig.canvas.draw()
        if not label_boxes_overlap(fig, texts):
            break
    assert not label_boxes_overlap(fig, texts), f"S17c labels overlap: {label_boxes_overlap(fig, texts)}"
save_figure(fig, "figS17c_observational_strategies")
plt.show()


## §6 Summary numbers for the text

Anchors (median over arms at λ = 0 and λ = 1) and the per-arm change from λ = 0 to λ = 1 of the
two combined axes. `r45_results_numbers.py` reads the slopes from `gradient_slopes.tsv`; this table
is for reading.

In [ ]:
# Per arm: mean at λ = 0, at λ = 1 and the change, for the two combined axes and N.
summary_scores = ["batch_removal", "biology_retention", "N", "L"]
anchor_rows = gradient_summary[gradient_summary["score"].isin(summary_scores) & gradient_summary["lambda"].isin([0.0, 1.0])]
anchor_table = anchor_rows.pivot_table(index=["arm", "score"], columns="lambda", values="value_mean").reset_index()
anchor_table.columns = ["arm", "score", "mean_at_lambda0", "mean_at_lambda1"]
anchor_table["change"] = anchor_table["mean_at_lambda1"] - anchor_table["mean_at_lambda0"]
anchor_table["method_label"] = anchor_table["arm"].map(arm_label)
save_table(anchor_table, "nb04_gradient_lambda0_lambda1")
print(anchor_table.pivot_table(index="method_label", columns="score", values="change").round(3).to_string())

## §7 Notes (gate G4 answered 2026-11-07)

- Figure 6 gradient panel: only the methods of the 22 selected approaches (G4.1); no harmonization,
  MNN and the other arms are in S17.
- Angel and M-ComBat were added to the gradient after the visual check selected them (pod run from
  the Mac, RUNBOOK §6.5); the Methods say so. Until their runs are collected they are absent from
  every panel.
- ComBat with the diagnosis covariate fails at λ = 1 (all 3 replicates) and on the
  single-diagnosis control: reported as an expected failure (§3e), not as a point. M-ComBat uses
  the same covariate and is expected to behave the same.
- N (leave-one-batch-out prediction) falls with V for every arm, including no harmonization:
  at λ = 1 a held-out batch carries a diagnosis that is rare or absent in the other batches, so
  part of the drop is a property of the metric, not of the methods.
- S17e: as confounding grows, no harmonization moves to the top ranks of the rule (above V ≈ 0.4
  almost the best): under strong confounding it is better not to harmonize (Discussion, G4.5).

**Figure review 2026-11-08** (`daniil_review_261108.md`):

- *S17a vs S17f — what differs, why both?* S17a shows every one of the 14 methods on six scores,
  with the uncapped curve and the single-diagnosis control: where the selected methods sit among
  all methods. S17f shows only no harmonization and the methods of the final selection on the
  same scores, without the controls, on a scale where their differences are visible: how the
  selected methods compare with each other.
- *Agreement with batch-free fold changes* (the former two S17a panels) is removed from the
  figures, Supplementary File 2 and the gradient statistics (r13 `SCORES`): the metric space
  stays at the 22-metric panel + L / N. Removing its rows from the BH family left the trend count
  at 43 of 78.
- *How were the "confidence intervals" made?* They are not confidence intervals: each band is
  mean ± SD of the 3 replicate draws at that λ. The slope CIs of S17b are the only CIs here.
- S17e: no band on either half; S17c: strategy colours, code labels without overlap.
